# Notebook 7 — Sweep en ε pour N=20 blocs

Reproduction du sweep en ε (Fig. similaire à Scholz) pour N = 20 blocs.

**ε balayés :** 0.02, 0.5, 2.0, 8.0, 12.0, 20.0  
**Marqueurs :** `'.-'` (ms=3) sur tous les tracés (demande Roxane)

**Intégrateur par ε :**
| ε       | Méthode | Raison                         |
|---------|---------|--------------------------------|
| ≤ 0.5   | RK45    | Système non-raide              |
| ≥ 2.0   | Radau   | Raideur ↑ (pics de vitesse)    |

**Note :** ε=2 et ε=8 nécessitent des durées plus longues (cluster EPFL)  
pour obtenir des exposants de Lyapunov fiables.

Paramètres : N=20, γ_μ=0.5, γ_λ=√0.2, ξ=0.5, f̃=3.2


In [1]:
"""
ε sweep for N=20 blocks — Erickson et al. (2011)
(version with markers '.-' sur tous les tracés)
"""

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc, time

N         = 20
GAMMA_MU  = 0.5
GAMMA_LAM = np.sqrt(0.2)
XI        = 0.5
F_TILDE   = 3.2
SIGMA     = 1.0

EPS_LIST  = [0.02, 0.5, 2.0, 8.0, 12.0, 20.0]

CONFIG = {
    0.02: dict(T=1500, dt=20, ms=2.0,  rtol=1e-5, atol=1e-7, method='RK45'),
    0.5:  dict(T=1500, dt=20, ms=2.0,  rtol=1e-5, atol=1e-7, method='RK45'),
    2.0:  dict(T=75,   dt=1,  ms=0.5,  rtol=1e-5, atol=1e-7, method='Radau'),
    8.0:  dict(T=75,   dt=1,  ms=0.5,  rtol=1e-6, atol=1e-8, method='Radau'),
    12.0: dict(T=100,  dt=1,  ms=0.5,  rtol=1e-6, atol=1e-8, method='Radau'),
    20.0: dict(T=1000, dt=5,  ms=0.5,  rtol=1e-6, atol=1e-8, method='Radau'),
}
EPS0 = 1e-8

def make_y0():
    u0b   = -F_TILDE * GAMMA_LAM**2 / (XI * GAMMA_MU**2)
    x_bar = np.array([(j - 0.5) * 20.0 / N for j in range(1, N + 1)])
    u_init = u0b + np.exp(-((x_bar - 10.0)**2) / SIGMA**2)
    return np.concatenate([u_init, np.zeros(N), np.zeros(N)])

def make_rhs(eps, law='slip'):
    gm2 = GAMMA_MU**2; gl2 = GAMMA_LAM**2
    def rhs(t, y):
        u  = y[:N]; v  = y[N:2*N]; Th = y[2*N:]
        ul  = np.concatenate([[u[0]],  u[:-1]])
        ur  = np.concatenate([u[1:],   [u[-1]]])
        vp1 = np.maximum(v + 1.0, 1e-15); lv = np.log(vp1)
        dv  = gm2*(ul-2*u+ur) - gl2*u - (gm2/XI)*(F_TILDE+Th+lv)
        if law == 'slip':
            dTh = -vp1*(Th + (1.+eps)*lv)
        else:  # aging
            eps1 = 1.+eps
            dTh = eps1*(np.exp(-Th/eps1) - vp1)
        return np.concatenate([v, dv, dTh])
    return rhs

def _ivp(rhs, t0, t1, y, ms, rtol, atol, method='auto'):
    if method == 'auto':
        v_max = np.abs(y[N:2*N]).max()
        method = 'Radau' if v_max > 0.5 else 'RK45'
    return solve_ivp(rhs, [t0, t1], y,
                     method=method, rtol=rtol, atol=atol,
                     max_step=ms, dense_output=False)

def simulate(eps, law='slip'):
    cfg = CONFIG[eps]
    T, dt, ms, rtol, atol = cfg['T'], cfg['dt'], cfg['ms'], cfg['rtol'], cfg['atol']
    rhs   = make_rhs(eps)
    c_idx = N // 2

    y0  = make_y0()
    y0p = y0.copy(); y0p[0] += EPS0

    y = y0.copy(); yp = y0p.copy()
    t_cur = 0.0; log_sum = 0.0

    t_traj=[]; v_c=[]; u_c=[]; Th_c=[]
    t_ly=[]; Lambda=[]
    t_wall = time.time()

    while t_cur < T - 1e-10:
        t_end = min(t_cur + dt, T)
        method_cfg = cfg['method']
        sol  = _ivp(rhs, t_cur, t_end, y,  ms, rtol, atol, method_cfg)
        solp = _ivp(rhs, t_cur, t_end, yp, ms, rtol, atol, method_cfg)

        if sol.status != 0 or solp.status != 0:
            print(f'    [WARN] solver failed at t={t_cur:.1f}')
            break

        y  = sol.y[:, -1]
        yp = solp.y[:, -1]

        # Stocker trajectoire — décimation pour garder ~8 pts/chunk
        stride = max(1, sol.t.size // 8)
        for k in range(0, sol.t.size, stride):
            t_traj.append(sol.t[k])
            v_c.append(sol.y[N + c_idx, k])
            u_c.append(sol.y[c_idx, k])
            Th_c.append(sol.y[2*N + c_idx, k])

        diff = yp - y; nd = np.linalg.norm(diff)
        if nd > 0:
            log_sum += np.log(nd / EPS0)
            yp = y + diff * (EPS0 / nd)

        t_cur = t_end
        if t_cur > 0:
            t_ly.append(t_cur)
            Lambda.append(log_sum / t_cur)

    wall = time.time() - t_wall
    L_final = Lambda[-1] if Lambda else 0.

    half  = len(Lambda) // 2
    t_h   = np.array(t_ly[half:])
    tL_h  = t_h * np.array(Lambda[half:])
    slope = np.polyfit(t_h, tL_h, 1)[0] if len(t_h) > 4 else 0.
    is_chaos = (slope > 0.005) and (L_final > 0.003)

    print(f'  ε={eps:5.2f}  T={T}  wall={wall:.0f}s  '
          f'npts_traj={len(t_traj)}  npts_lya={len(t_ly)}  '
          f'Λ={L_final:.5f}  → {"CHAOS" if is_chaos else "periodic"}')

    return (np.array(t_traj), np.array(v_c), np.array(u_c), np.array(Th_c),
            np.array(t_ly),   np.array(Lambda), is_chaos, L_final)

def plot_sweep(results):
    n_eps  = len(EPS_LIST)
    cmap   = plt.cm.plasma
    colors = [cmap(i / (n_eps - 1)) for i in range(n_eps)]

    fig, axes = plt.subplots(n_eps, 4, figsize=(18, 3.2 * n_eps))
    fig.suptitle(
        r'N=20 blocks — sweep en $\varepsilon$   '
        r'[$\gamma_\mu=0.5,\ \gamma_\lambda=\sqrt{0.2},\ \xi=0.5,\ \tilde{f}=3.2$]'
        '\nCentral block — steady state (2nd half)\n'
        r'$\Lambda(t)$ : two-trajectory method (Benettin 1980,  $\varepsilon_0=10^{-8}$)',
        fontsize=10, fontweight='bold'
    )

    col_titles = [
        r'Vitesse relative $\bar{v}_c$',
        r'Position relative $\bar{u}_c$',
        r'$\Lambda(t) = \frac{1}{t}\sum\ln\frac{\|\delta_i\|}{\varepsilon_0}$',
        r'State $\bar{\Theta}_c$',
    ]
    for c, title in enumerate(col_titles):
        axes[0, c].set_title(title, fontsize=9)

    for row, (eps, col) in enumerate(zip(EPS_LIST, colors)):
        (t_traj, v_c, u_c, Th_c,
         t_ly, Lambda, is_chaos, L_final) = results[eps]

        T      = CONFIG[eps]['T']
        t_half = T / 2
        mask_t = t_traj >= t_half
        mask_l = t_ly   >= t_half

        t_p  = t_traj[mask_t]
        v_p  = v_c[mask_t]; u_p = u_c[mask_t]; Th_p = Th_c[mask_t]
        t_L  = t_ly[mask_l]; L_p = Lambda[mask_l]

        n_traj = mask_t.sum()
        n_lya  = mask_l.sum()

        regime = ('CHAOS  Λ→{:.4f}'.format(L_final) if is_chaos
                  else 'periodic  Λ→0  ({:.4f})'.format(L_final))
        row_lbl = (rf'$\varepsilon={eps}$' + '\n' + regime
                   + f'\n({n_traj} pts traj, {n_lya} pts Λ)')

        MS = 3

        def ax(c): return axes[row, c]

        ax(0).plot(t_p, v_p, '.-', color=col, lw=0.6, markersize=MS)
        ax(0).axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
        ax(0).set_ylabel(row_lbl, fontsize=8, rotation=0,
                         labelpad=110, va='center', color=col)

        ax(1).plot(t_p, u_p, '.-', color=col, lw=0.6, markersize=MS)

        if len(t_L) > 1:
            ax(2).plot(t_L, L_p, '.-', color=col, lw=1.0, markersize=MS)
            ax(2).axhline(0, color='black', lw=0.8, ls='--')
            ax(2).fill_between(t_L, 0, L_p, where=(L_p > 0),
                               color='red',   alpha=0.15)
            ax(2).fill_between(t_L, 0, L_p, where=(L_p <= 0),
                               color='green', alpha=0.15)
            ax(2).axhline(L_final, color=col, lw=0.7, ls=':', alpha=0.8)
        ax(2).set_ylabel(r'$\Lambda$', fontsize=8)

        ax(3).plot(t_p, Th_p, '.-', color=col, lw=0.6, markersize=MS)
        ax(3).axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)

        for c in range(4):
            axes[row, c].grid(True, alpha=0.2)
            axes[row, c].tick_params(labelsize=7)
            if row == n_eps - 1:
                axes[row, c].set_xlabel(r'$\bar{t}$', fontsize=8)

        del t_traj, v_c, u_c, Th_c, t_ly, Lambda
        gc.collect()

    plt.tight_layout()
    out = 'bk_eps_sweep_N20.png'
    plt.savefig(out, dpi=140, bbox_inches='tight')
    plt.close(fig)
    print(f'\nFigure → {out}')

if __name__ == '__main__':
    print('Sweep ε — N=20 blocks  (Benettin 1980)\n')
    results = {}
    for eps in EPS_LIST:
        print(f'ε = {eps}')
        results[eps] = simulate(eps)
    plot_sweep(results)

Sweep ε — N=20 blocs  (Benettin 1980)

ε = 0.02
  ε= 0.02  T=1500  wall=0s  npts_traj=852  npts_lya=75  Λ=-0.01964  → périodique
ε = 0.5
  ε= 0.50  T=1500  wall=1s  npts_traj=677  npts_lya=75  Λ=0.01571  → CHAOS
ε = 2.0
  ε= 2.00  T=75  wall=1s  npts_traj=446  npts_lya=75  Λ=0.03675  → périodique
ε = 8.0
  ε= 8.00  T=75  wall=2s  npts_traj=464  npts_lya=75  Λ=0.16363  → CHAOS
ε = 12.0
  ε=12.00  T=100  wall=1s  npts_traj=561  npts_lya=100  Λ=0.04219  → CHAOS
ε = 20.0
  ε=20.00  T=1000  wall=51s  npts_traj=1926  npts_lya=200  Λ=0.88787  → CHAOS

Figure → bk_eps_sweep_N20.png


---
## Extension — Sweep en ε : Slip law vs Aging law

On refait le sweep en ε avec la **aging law** et on compare les séries temporelles
côte-à-côte avec la slip law.

**Hypothèse :** l'aging law devrait décaler le seuil de chaos vers des ε plus grands
(stabilisation par guérison intersismique).

In [2]:
"""
Extension — Slip law vs Aging law comparison (N=20 blocks).

Solution for eps>=8 (aging law) :
  Le contrôleur de pas adaptatif (LSODA, Radau) bloque car l'aging law
  creates extreme derivatives during resticking (exp(-Th/eps1) >> 1).
  Fix : forcer max_step=1e-3 avec Radau + tolerances relachees (rtol=1e-3).
  Le solveur n'adapte plus son pas → temps de calcul fixe et previsible.

Temps estimé sur PC Windows (~6 min/eps pour eps>=8, T=75).
"""
import numpy as np, gc, time, warnings
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def make_rhs_aging_local(eps):
    eps1 = 1. + eps
    gm2  = GAMMA_MU**2; gl2 = GAMMA_LAM**2
    def rhs(t, y):
        u  = y[:N]; v = y[N:2*N]; Th = y[2*N:]
        ul = np.r_[u[0], u[:-1]]; ur = np.r_[u[1:], u[-1]]
        vp1 = np.maximum(v + 1., 1e-15); lv = np.log(vp1)
        dv  = gm2*(ul-2*u+ur) - gl2*u - (gm2/XI)*(F_TILDE+Th+lv)
        # Clip a 10 : borne physique (ln(v_max)~4 pour v_max~80)
        dTh = eps1 * (np.exp(np.minimum(-Th/eps1, 10.)) - vp1)
        return np.concatenate([v, dv, dTh])
    return rhs


EPS_COMPARE = [0.5, 2.0, 8.0, 20.0]
COLORS = {'slip': '#1D9E75', 'aging': '#D85A30'}

# Parametres par eps pour l'aging law
# eps<=2 : LSODA adaptatif (rapide)
# eps>=8 : Radau + max_step force (contourne le controleur de pas qui bloque)
AGING_CFG = {
    0.5:  dict(T=300, method='LSODA', rtol=1e-5, atol=1e-7, ms=2.0,   forced=False),
    2.0:  dict(T=25,  method='LSODA', rtol=1e-5, atol=1e-7, ms=2.0,   forced=False),
    8.0:  dict(T=75,  method='Radau', rtol=1e-3, atol=1e-4, ms=1e-3,  forced=True),
    20.0: dict(T=75,  method='Radau', rtol=1e-3, atol=1e-4, ms=1e-3,  forced=True),
}

fig, axes = plt.subplots(len(EPS_COMPARE), 2,
                         figsize=(13, 4*len(EPS_COMPARE)),
                         sharex='row', sharey='row')
fig.suptitle(
    'N=20 — Slip law vs Aging law\n'
    'eps>=8 : Radau pas force (max_step=1e-3, rtol=1e-3) — ~6 min/eps sur PC',
    fontsize=11, fontweight='bold')
axes[0, 0].set_title('Slip law (Ruina 1983)',    fontsize=10, fontweight='bold')
axes[0, 1].set_title('Aging law (Dieterich 1979)', fontsize=10, fontweight='bold')

for row, eps in enumerate(EPS_COMPARE):
    cfg_slip = CONFIG[eps]
    cfg_age  = AGING_CFG[eps]

    for col, law in enumerate(['slip', 'aging']):
        if law == 'slip':
            T_run  = cfg_slip['T']
            method = cfg_slip['method']
            ms     = cfg_slip['ms']
            rtol   = cfg_slip['rtol']; atol = cfg_slip['atol']
            rhs_f  = make_rhs(eps, 'slip')
        else:
            T_run  = cfg_age['T']
            method = cfg_age['method']
            ms     = cfg_age['ms']
            rtol   = cfg_age['rtol']; atol = cfg_age['atol']
            rhs_f  = make_rhs_aging_local(eps)

        forced_str = ' [pas force]' if (law=='aging' and cfg_age.get('forced')) else ''
        print(f'  eps={eps}, {law} (T={T_run}, {method}{forced_str})...',
              end=' ', flush=True)
        t0c = time.time()
        y0  = make_y0()

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            sol = solve_ivp(rhs_f, [EPS0, T_run], y0,
                            method=method, rtol=rtol, atol=atol,
                            max_step=ms, dense_output=False)

        t_all = sol.t; v_mid = sol.y[N + N//2, :]
        mask  = t_all >= T_run / 2
        axes[row, col].plot(t_all[mask], v_mid[mask], '.-',
                            color=COLORS[law], lw=0.5, ms=1.5, alpha=0.85)
        axes[row, col].set_ylabel(rf'$\bar{{v}}_c$, $\varepsilon={eps}$', fontsize=9)
        axes[row, col].grid(True, ls=':', alpha=0.35)
        print(f'{time.time()-t0c:.1f}s — {len(sol.t)} pts')
        del sol; gc.collect()

for col in range(2):
    axes[-1, col].set_xlabel(r'$\bar{t}$', fontsize=9)

plt.tight_layout()
plt.savefig('nb7_eps_sweep_aging_vs_slip.png', dpi=140, bbox_inches='tight')
plt.close(fig)
print('\nFigure → nb7_eps_sweep_aging_vs_slip.png')

  eps=0.5, slip (T=1500, RK45)... 0.5s — 4362 pts
  eps=0.5, aging (T=300, LSODA)... 0.7s — 4144 pts
  eps=2.0, slip (T=75, Radau)... 0.3s — 338 pts
  eps=2.0, aging (T=25, LSODA)... 0.0s — 214 pts
  eps=8.0, slip (T=75, Radau)... 0.7s — 840 pts
  eps=8.0, aging (T=75, Radau [pas force])... 

KeyboardInterrupt: 